# PHYS 690 - Lecture 15: Monte Carlo Sampling and Integration

## One damped Ramsey experiment, many random samples

Unit 4 will follow one model across five lectures: a two-level quantum system measured after a variable delay. Its damped oscillation is deliberately close to the quantum-optics model used in Lecture 6.  In a Ramsey-type measurement, repeated preparation and measurement produce binary outcomes. The excited-state probability is modeled as

$$
p_e(t \mid \boldsymbol{\theta}) = \frac{1}{2}\left[1 + C e^{-t/\tau}\cos(\omega t + \phi)\right],
$$

where $\boldsymbol{\theta} = (C, \tau, \omega, \phi)$ contains the contrast, coherence time, angular frequency, and phase. This model appears in qubit calibration and quantum sensing; see the [Qiskit Experiments RamseyXY documentation](https://qiskit-community.github.io/qiskit-experiments/stubs/qiskit_experiments.library.characterization.RamseyXY.html) and the open review [Degen, Reinhard, and Cappellaro, *Quantum sensing*](https://arxiv.org/abs/1611.02427).

## Learning goals and class plan

By the end of class, you should be able to:

1. explain what a pseudorandom-number generator and seed do;
2. express an integral as an expectation value;
3. explain when Monte Carlo integration is useful as the number of variables grows;
4. construct Monte Carlo estimators and their standard errors;
5. verify the characteristic $1/\sqrt{N}$ sampling-error scaling; and
6. use stratified sampling to reduce variance without changing the target average.

In [ ]:
# NumPy handles arrays and random draws; pandas labels results; Matplotlib plots;
# SciPy provides the quadrature benchmark used later.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import quad

# Apply consistent styling to all figures in the notebook.
plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

## Why use Monte Carlo integration in physics?

Many useful physics quantities are averages over several variables. Examples include averages over position and momentum in phase space, averages over uncertain experimental conditions, and integrating a detector response over the particles that can reach the apparatus.

For one or two variables, a regular grid and numerical quadrature are often simple and efficient. But if we use $m$ grid points along each of $d$ dimensions, the grid contains $m^d$ evaluations. That cost grows rapidly as more variables are added. Monte Carlo integration instead draws representative points from the relevant ranges or distributions and averages the function values. Its characteristic error decreases like $1/\sqrt{N}$, where $N$ is the number of random samples; the rate does not acquire an additional power of $N$ just because the number of dimensions increases.

Monte Carlo is not automatically more accurate or faster in every problem. The variance of the sampled values still matters, and low-dimensional quadrature is usually preferable when it is available. We will begin with an easy one-dimensional Ramsey example to understand the estimator, then extend the same probability model to a two-dimensional average over delay and detuning.


## Part 1: A reproducible stream of pseudorandom numbers

A computer normally produces **pseudorandom** values: a deterministic algorithm maps an internal state to a sequence that passes useful statistical tests. A seed initializes that state. Reusing a seed reproduces a sequence, which is valuable for debugging and teaching, but it does not make the outcomes physically deterministic.

Use NumPy's `Generator` interface rather than the older global random functions. One generator should usually be created near the beginning of a calculation and passed explicitly when needed.

**Discuss:** When is a fixed seed scientifically helpful? When might reporting only one seed hide a fragile result?

In [ ]:
# Create two independent random number generators with the same seed.
rng_a = np.random.default_rng(690)
rng_b = np.random.default_rng(690)

# Identical seeds and identical calls reproduce the same sequence.
# Store matching draws in a labeled table so their equality is easy to inspect.
reproducibility_df = pd.DataFrame({
    "draw_from_a": rng_a.random(6),
    "draw_from_b": rng_b.random(6),
})
reproducibility_df["difference"] = (
    reproducibility_df["draw_from_a"] - reproducibility_df["draw_from_b"]
)
reproducibility_df

### The repeated physical model

We use normalized time in microseconds and angular frequency in $\mathrm{rad}/\mu\mathrm{s}$. The probability remains between zero and one because $0 \leq C \leq 1$. These parameter values will be reused throughout Unit 4.

In [ ]:
# Define the shared Ramsey probability model.
def ramsey_probability(t, contrast, tau, omega, phase):
    # The exponential envelope models the decay of fringe contrast over time.
    envelope = np.exp(-np.asarray(t) / tau)
    # The cosine gives the Ramsey oscillation at angular frequency omega.
    oscillation = np.cos(omega * np.asarray(t) + phase)
    # Shift and scale the oscillation so the result is an excited-state probability.
    return 0.5 * (1.0 + contrast * envelope * oscillation)


# Fix a shared generating parameter set for the lecture examples.
true_parameters = {
    "contrast": 0.78,
    "tau": 2.40,              # microseconds
    "omega": 2.0 * np.pi * 0.85,  # rad/microsecond
    "phase": 0.25,            # rad
}

# A dense time grid makes the model curve appear smooth.
time_plot = np.linspace(0.0, 5.0, 600)
# Keep plotted coordinates and probabilities together in a labeled DataFrame.
model_df = pd.DataFrame({
    "time_us": time_plot,
    "excited_probability": ramsey_probability(time_plot, **true_parameters),
})

fig, ax = plt.subplots()
ax.plot(model_df["time_us"], model_df["excited_probability"], lw=2)
ax.set(xlabel=r"Delay $t$ ($\mu$s)", ylabel=r"$p_e(t)$",
       title="Damped Ramsey probability")
ax.set_ylim(0.0, 1.0)
plt.show()

## Part 2: Emulating a Ramsey scan with Monte Carlo sampling

### What is a Ramsey measurement?

A Ramsey measurement uses two short control pulses to turn a phase into a measurable population. In the review's first Ramsey example, the sequence is:

1. Initialize the two-level sensor in its lower-energy state $|0\rangle$.
2. Apply a $\pi/2$ pulse to prepare the equal superposition $\left(|0\rangle+|1\rangle\right)/\sqrt{2}$.
3. Let the system evolve freely for a chosen delay $t$. The energy splitting, including any signal-induced shift, causes the two parts of the superposition to accumulate a relative phase.
4. Apply a second $\pi/2$ pulse. It converts the accumulated phase into a population difference that can be measured.
5. Read out whether the sensor is in $|0\rangle$ or $|1\rangle$, then repeat the sequence many times.

The review's ideal Ramsey model produces oscillations called **Ramsey fringes** as the delay changes. Here we use a damped, flexible form for the excited-state probability,

$$
p_e(t\mid\boldsymbol{\theta})=\frac{1}{2}\left[1+C e^{-t/\tau}\cos(\omega t+\phi)\right].
$$

$C$ sets the fringe contrast, $\tau$ describes the decay of coherence, $\omega$ sets the fringe frequency, and $\phi$ sets the phase relative to the chosen time and drive reference. This is a phenomenological extension of the ideal sequence: reduced contrast can summarize imperfect preparation, control, and readout, while the decaying envelope models loss of coherence. The sequence and its ideal transition probability are described in Degen, Reinhard, and Cappellaro, [*Quantum sensing*, Section IV.C, “First example: Ramsey measurement”](https://arxiv.org/abs/1611.02427).

At a fixed delay $t$, each measurement is a binary observation:

$$
X_j=
\begin{cases}
1, & \text{the sensor is measured in }|1\rangle,\\
0, & \text{the sensor is measured in }|0\rangle.
\end{cases}
$$

If the sequence is repeated $n_{\mathrm{shots}}$ times at that delay, the observed fraction

$$
\widehat{p}_e(t)=\frac{1}{n_{\mathrm{shots}}}\sum_{j=1}^{n_{\mathrm{shots}}}X_j
$$

estimates the probability $p_e(t)$. Repeating this at several delays maps out the Ramsey fringes.

### What quantity will Monte Carlo estimate?

Now imagine measuring the excited-state probability over a delay window $a < t < b$. At each chosen delay, repeated Ramsey shots estimate the excited-state fraction. We want to compare the average excitation predicted by the model across this window with the corresponding measured behavior.

The model predicts a probability at every delay. Integrating its curve gives the area under the theoretical probability curve:

$$
I=\int_a^b p_e(t)\,dt.
$$

This area has units of time, so it is not itself a probability. Dividing both sides by the window width gives the average predicted excited-state probability,

$$
\overline{p}_e=\frac{I}{b-a}=\frac{1}{b-a}\int_a^b p_e(t)\,dt.
$$

For a delay $T$ drawn uniformly from $[a,b]$, its probability density is $1/(b-a)$. Thus, the average of $p_e(T)$ over repeated delay draws is exactly $\overline{p}_e$; in probability notation, $\mathbb{E}[p_e(T)]$ means this average over the possible values of $T$.

This average is the direct comparison when delays are selected uniformly across the window and shots are pooled with equal weight. If an experiment uses a different delay schedule or different numbers of shots at each delay, the comparison should instead use the actual sampling and weighting scheme.

Draw $N$ random delays $T_i$ uniformly from $[a,b]$, evaluate the model probability at each delay, and divide the sum by $N$:

$$
\widehat{\overline{p}}_{e,N}=\frac{1}{N}\sum_{i=1}^{N}p_e(T_i).
$$

The corresponding estimate of the area, if needed, is this average multiplied by $b-a$. Here $N$ counts sampled delays. This calculation averages model probabilities rather than drawing the binary readouts $X_j$; that isolates numerical integration error from detector shot noise. Lecture 16 will explicitly simulate the binary outcomes and their additional variability.

### How uncertain is the Monte Carlo estimate?

Let $s_p$ be the sample standard deviation of the $N$ sampled probability values $p_e(T_i)$:

$$
s_p=\sqrt{\frac{1}{N-1}\sum_{i=1}^{N}\left[p_e(T_i)-\widehat{\overline{p}}_{e,N}\right]^2}.
$$

The **standard error** estimates how much the average-probability estimate would typically vary if we repeated the random-delay calculation. Dividing the integral standard error by $b-a$ gives

$$
\mathrm{SE}\!\left(\widehat{\overline{p}}_{e,N}\right)=\frac{s_p}{\sqrt{N}}.
$$

This is Monte Carlo sampling uncertainty. It is not uncertainty in the physical parameters, and it does not include the randomness of the binary detector outcomes.

### Why the estimator works

If the delays are drawn uniformly from $[a,b]$, the average of the sampled values $p_e(T_i)$ approaches the average value of the probability curve over that interval. This sample mean directly estimates $\overline{p}_e$; multiplying by $b-a$ would recover the area $I$.

The estimator is **unbiased**: over many independent repetitions of the random-delay calculation, its average equals the target average probability. A single estimate will generally differ from $\overline{p}_e$. Its typical variation is described by the standard error above, which decreases in proportion to $1/\sqrt{N}$ when the sampled values are independent and have finite variance.

For a sufficiently large sample, the estimate's deviations from the target are often approximately described by a normal distribution. This approximation can be poor for small samples, rare-event problems, or cases where the sampled values have very large variation. Repeating the calculation with independent random seeds provides a useful check.

In [ ]:
# Define the finite delay window and a scalar probability function for quadrature.
t_min, t_max = 0.0, 5.0  # microseconds

def probability_at_time(t):
    return ramsey_probability(t, **true_parameters)

# Use adaptive quadrature as a high-accuracy area benchmark.
reference_integral, quadrature_error = quad(probability_at_time, t_min, t_max)
# Divide by the delay-window width because the target is an average probability.
reference_average = reference_integral / (t_max - t_min)

print(f"Reference average excitation probability = {reference_average:.8f}")
print(f"(Area under probability curve = {reference_integral:.8f} microseconds)")
print(f"quad error estimate for area = {quadrature_error:.2e}")

### Observe the error scaling

Monte Carlo is not the best algorithm for this smooth one-dimensional integral; quadrature is faster. Its strength is that the same estimator generalizes to high-dimensional integrals and complicated sampling regions. Here the easy benchmark lets us inspect the method honestly.

In [ ]:
def direct_mc_average(n_samples, rng):
    # Draw delays uniformly, matching the average over the selected window.
    sampled_t = rng.uniform(t_min, t_max, size=n_samples)
    # Evaluate the model at every sampled delay to get the values being averaged.
    sampled_values = probability_at_time(sampled_t)

    # The sample mean estimates average excitation probability; its standard
    # error is the sample spread divided by the square root of the sample count.
    estimate = np.mean(sampled_values)
    standard_error = np.std(sampled_values, ddof=1) / np.sqrt(n_samples)
    return estimate, standard_error


# Repeat the estimate at increasing sample sizes with one reproducible stream.
scaling_rng = np.random.default_rng(1501)
sample_sizes = np.array([100, 300, 1000, 3000, 10000, 30000])
scaling_rows = []

for n_samples in sample_sizes:
    estimate, standard_error = direct_mc_average(n_samples, scaling_rng)
    # Save the estimate, its distance from the benchmark, and its estimated SE.
    scaling_rows.append({
        "n_samples": n_samples,
        "average_probability": estimate,
        "absolute_error": abs(estimate - reference_average),
        "estimated_standard_error": standard_error,
    })

# Organize estimates in a table for plotting and comparison.
scaling_df = pd.DataFrame(scaling_rows)

# The left panel checks the benchmark; the right checks the expected 1/sqrt(N) trend.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].errorbar(scaling_df["n_samples"], scaling_df["average_probability"],
                 yerr=scaling_df["estimated_standard_error"], fmt="o-")
axes[0].axhline(reference_average, color="black", ls="--", label="quadrature")
axes[0].set(xscale="log", xlabel="Number of samples",
            ylabel="Average excitation probability")
axes[0].legend()

axes[1].loglog(scaling_df["n_samples"],
               scaling_df["estimated_standard_error"], "o-", label="estimated SE")
reference_line = scaling_df["estimated_standard_error"].iloc[0] * np.sqrt(
    scaling_df["n_samples"].iloc[0] / scaling_df["n_samples"]
)
axes[1].loglog(scaling_df["n_samples"], reference_line, "--", label=r"$1/\sqrt{N}$")
axes[1].set(xlabel="Number of samples", ylabel="Estimated standard error")
axes[1].legend()
fig.suptitle("Monte Carlo estimate of average excitation probability")
fig.tight_layout()
plt.show()

## Part 3: In-class coding activity 1 - Estimate the average probability

Use $N = 4000$ independent uniform delay samples to estimate $\overline{p}_e$. Report the estimate, its standard error, and the approximate 1-sigma interval $\widehat{\overline{p}}_{e,N} \pm \mathrm{SE}(\widehat{\overline{p}}_{e,N})$. Compare the quadrature benchmark with this interval and calculate the difference between the estimate and benchmark in units of the Monte Carlo standard error.

When the estimate's sampling distribution is approximately Gaussian, about 68.3% of repeated estimates fall within one standard error of the true average probability. Thus, a benchmark just outside one particular 1-sigma interval is not by itself evidence of a coding error or a meaningful disagreement; about 31.7% of such intervals miss the true value.

Before coding, identify which line represents randomness and which line represents the deterministic physical model.

In [ ]:
# Fix the activity random stream and sample count for reproducible results.
activity_rng = np.random.default_rng(1515)
activity_n = 4000
# TODO: Draw activity_n uniform delays over the delay window.
activity_t = ...
# TODO: Evaluate the Ramsey probability at every sampled delay.
activity_values = ...
# TODO: Calculate the sample mean as the estimated average probability.
activity_estimate = ...
# TODO: Estimate its standard error using the sample standard deviation.
activity_standard_error = ...
# TODO: Form the lower and upper limits of an approximate 1-sigma interval.
activity_interval = ...
# TODO: Express the estimate-minus-benchmark difference in standard-error units.
activity_difference_in_se = ...

# Collect the estimate, uncertainty interval, and benchmark comparison in one table.
activity_summary_df = pd.DataFrame({
    "quantity": ["average probability estimate", "standard error", "1-sigma lower", "1-sigma upper", "quadrature benchmark", "estimate minus benchmark (SE units)"],
    "value": [activity_estimate, activity_standard_error,
              activity_interval[0], activity_interval[1], reference_average,
              activity_difference_in_se],
})
activity_summary_df

## Part 4: A two-dimensional Ramsey average

The one-dimensional example averages over delay. A real experiment can also have a spread in detuning: different trials may experience a small offset $\Delta$ from the nominal angular frequency $\omega$. For this worked example, suppose both the delay and detuning are sampled uniformly over finite ranges. The detuning range is a simple model of a controlled calibration window.

For a trial with delay $t$ and detuning $\Delta$, use

$$
p_e(t,\Delta)=\frac{1}{2}\left[1+C e^{-t/\tau}\cos\!\left((\omega+\Delta)t+\phi\right)\right].
$$

The quantity we want is the average predicted excited-state probability across the rectangle $a\leq t\leq b$ and $-\Delta_{\max}\leq\Delta\leq\Delta_{\max}$:

$$
\overline{p}_e=
\frac{1}{(b-a)(2\Delta_{\max})}
\int_a^b\!dt\int_{-\Delta_{\max}}^{\Delta_{\max}}\!d\Delta\;p_e(t,\Delta).
$$

Each Monte Carlo sample is now a **pair** $(t_i,\Delta_i)$. The estimator is the average probability across the sampled pairs,

$$
\widehat{\overline{p}}_{e,N}=\frac{1}{N}\sum_{i=1}^{N}p_e(t_i,\Delta_i).
$$

If $s_p$ is the sample standard deviation of these $N$ probability values, the estimated standard error on this average is $s_p/\sqrt{N}$. Multiplying the average and its standard error by the rectangle area gives the corresponding integral estimate and its standard error.

The binary measurement outcome is still not simulated here; this calculation estimates an average probability. Lecture 16 will separately model binary shot outcomes and correlated calibration uncertainty.

In two dimensions, a midpoint grid is still feasible and serves as a useful reference. The Monte Carlo example demonstrates how to sample jointly and estimate uncertainty; it does not claim that random sampling is the fastest method for this small problem.


In [ ]:
# Set the finite time and detuning ranges for the 2D average.
detuning_half_width = 0.35  # rad/microsecond
time_width = t_max - t_min
detuning_width = 2.0 * detuning_half_width
rectangle_area = time_width * detuning_width


# Evaluate the probability on a midpoint grid as a deterministic reference.
n_time_grid = 160
n_detuning_grid = 140
# Use cell centers for a midpoint-rule reference grid in both dimensions.
time_midpoints = t_min + (np.arange(n_time_grid) + 0.5) * time_width / n_time_grid
detuning_midpoints = (
    -detuning_half_width
    + (np.arange(n_detuning_grid) + 0.5) * detuning_width / n_detuning_grid
)
# Build coordinate arrays so the model is evaluated at every grid location.
time_mesh, detuning_mesh = np.meshgrid(time_midpoints, detuning_midpoints)

# Evaluate the damped Ramsey probability over the complete 2D grid.
grid_probabilities = 0.5 * (
    1.0
    + true_parameters["contrast"]
    * np.exp(-time_mesh / true_parameters["tau"])
    * np.cos(
        (true_parameters["omega"] + detuning_mesh) * time_mesh
        + true_parameters["phase"]
    )
)
# Average grid values to estimate the rectangle average; multiply by area for the integral.
grid_average_probability = np.mean(grid_probabilities)
grid_integral = rectangle_area * grid_average_probability

# Store the grid in a DataFrame so its coordinates and values stay labeled.
probability_surface_df = pd.DataFrame({
    "time_us": time_mesh.ravel(),
    "detuning_rad_us": detuning_mesh.ravel(),
    "excited_probability": grid_probabilities.ravel(),
})
surface_pivot_df = probability_surface_df.pivot(
    index="detuning_rad_us", columns="time_us", values="excited_probability"
)


# Draw independent uniform (delay, detuning) pairs over the rectangle.
two_d_rng = np.random.default_rng(1520)
two_d_n = 12000
two_d_samples_df = pd.DataFrame({
    "time_us": two_d_rng.uniform(t_min, t_max, size=two_d_n),
    "detuning_rad_us": two_d_rng.uniform(
        -detuning_half_width, detuning_half_width, size=two_d_n
    ),
})
two_d_samples_df["excited_probability"] = 0.5 * (
    1.0
    + true_parameters["contrast"]
    * np.exp(-two_d_samples_df["time_us"] / true_parameters["tau"])
    * np.cos(
        (true_parameters["omega"] + two_d_samples_df["detuning_rad_us"])
        * two_d_samples_df["time_us"]
        + true_parameters["phase"]
    )
)

# The Monte Carlo average is the sample mean; estimate its SE from sample spread.
two_d_average = two_d_samples_df["excited_probability"].mean()
two_d_average_se = (
    two_d_samples_df["excited_probability"].std(ddof=1) / np.sqrt(two_d_n)
)
two_d_integral = rectangle_area * two_d_average
two_d_integral_se = rectangle_area * two_d_average_se

# Compare the midpoint grid and Monte Carlo results in one table.
two_d_summary_df = pd.DataFrame({
    "method": ["midpoint grid", "Monte Carlo"],
    "average_probability": [grid_average_probability, two_d_average],
    "integral_over_rectangle": [grid_integral, two_d_integral],
    "integral_standard_error": [np.nan, two_d_integral_se],
})
display(two_d_summary_df)

# Plot the probability surface and overlay a subset of sampled coordinate pairs.
fig, ax = plt.subplots(figsize=(8.5, 4.8))
image = ax.pcolormesh(
    surface_pivot_df.columns.to_numpy(),
    surface_pivot_df.index.to_numpy(),
    surface_pivot_df.to_numpy(),
    shading="auto",
    cmap="viridis",
)
sampled_points_df = two_d_samples_df.iloc[:300]
ax.scatter(sampled_points_df["time_us"], sampled_points_df["detuning_rad_us"],
           s=9, facecolors="none", edgecolors="white", linewidths=0.5,
           label="first 300 Monte Carlo samples")
ax.set_xlabel(r"Delay $t$ ($\mu$s)")
ax.set_ylabel(r"Detuning offset $\Delta$ (rad/$\mu$s)")
ax.set_title(r"Ramsey probability across delay and detuning")
fig.colorbar(image, ax=ax, label=r"$p_e(t,\Delta)$")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

print(f"Grid average probability = {grid_average_probability:.6f}")
print(f"Monte Carlo average probability = {two_d_average:.6f} +/- {two_d_average_se:.6f}")
print(f"Grid integral = {grid_integral:.6f}")
print(f"Monte Carlo integral = {two_d_integral:.6f} +/- {two_d_integral_se:.6f}")


### How the same idea extends to more dimensions

For $d$ variables, one Monte Carlo sample is a point with $d$ coordinates, and the estimator is still the average of the integrand evaluated at sampled points. A regular grid with $m$ points along each coordinate requires $m^d$ evaluations, while Monte Carlo uses the chosen number $N$ of joint samples. Its standard error still scales as $1/\sqrt{N}$, although the variance of the sampled integrand can become large in a difficult high-dimensional problem. Sampling from physically relevant distributions and using variance-reduction methods can help.

Examples where the integral may involve many variables include:

- **Plasma physics:** average collision or reaction rates over particle velocities, collision angles, and local plasma conditions; integrate a distribution function over position and velocity space.
- **Quantum optics:** average photon-count probabilities over fluctuating laser intensity, detuning, phase, polarization, and detector timing jitter.
- **Nuclear and particle physics:** integrate a differential cross section over multidimensional final-state phase space, or average detector response over particle momenta, production positions, and detector variables.

The 2D Ramsey calculation is small enough that midpoint quadrature remains practical. These higher-dimensional examples show why the same sampling-and-average strategy becomes valuable when a grid would require too many evaluations.


## Part 5: Variance reduction by stratified sampling

Direct sampling can leave accidental gaps and clusters. **Stratified sampling** divides the interval into $N$ equal pieces and draws one random point from each. The estimator remains an average over the full interval, but the sample covers the domain more evenly.

For stratum $i$,

$$
T_i = a + (b-a)\frac{i+U_i}{N}, \qquad i=0,1,\ldots,N-1,
\qquad U_i \sim \mathrm{Uniform}(0,1).
$$

The index starts at zero so the first draw lies in the first subinterval and the final draw lies in the last one.

This does not guarantee lower variance for every integrand, but it is effective for many smooth functions. It is also a bridge to quasi-Monte Carlo methods, which construct deliberately space-filling point sets.

### In-class coding activity 2 - Compare two sampling designs

Complete the stratified estimator. The following completed cell will repeat both estimators 250 times at the same sample size, so the spread across repetitions measures their actual sampling variability.

In [ ]:
def stratified_mc_average(n_samples, rng):
    # Draw one point uniformly from each equal stratum of the unit interval.
    unit_interval_samples = np.empty(n_samples)
    for i in range(n_samples):
        stratum_start = i / n_samples
        stratum_end = (i + 1) / n_samples
        unit_interval_samples[i] = rng.uniform(stratum_start, stratum_end)

    # TODO: Rescale unit-interval points to the physical delay window [t_min, t_max].
    sampled_t = ...
    # Evaluate the model at those stratified delays and average the probabilities.
    # TODO: Complete the average-probability estimate.
    estimate = ...
    return estimate

   
# Test stratified Monte Carlo
print(f"Stratified Monte Carlo average probability = {stratified_mc_average(20, activity_rng):.6f}")

In [ ]:
# This comparison assumes the activity function above is complete.
comparison_rng = np.random.default_rng(1516)
comparison_n = 256
n_repetitions = 250
# Store one result per method and repetition for the spread comparison.
comparison_rows = []

# Repeat both designs at the same sample count to compare sampling variability.
for repetition in range(n_repetitions):
    direct_estimate, _ = direct_mc_average(comparison_n, comparison_rng)
    stratified_estimate = stratified_mc_average(comparison_n, comparison_rng)
    comparison_rows.extend([
        {"method": "independent uniform", "average_probability": direct_estimate},
        {"method": "stratified", "average_probability": stratified_estimate},
    ])

# Keep repeated estimates in a table, then summarize their mean and empirical SD.
comparison_df = pd.DataFrame(comparison_rows)
method_summary_df = comparison_df.groupby("method")["average_probability"].agg(
    mean="mean", empirical_standard_deviation="std"
).reset_index()
display(method_summary_df)

# Overlay the estimate distributions; narrower spread means lower variability here.
fig, ax = plt.subplots()
for method, group in comparison_df.groupby("method"):
    ax.hist(group["average_probability"], bins=24, alpha=0.55, density=True, label=method)
ax.axvline(reference_average, color="black", ls="--", label="benchmark")
ax.set(xlabel="Average excitation probability estimate", ylabel="Density",
       title=f"Repeated estimates with N={comparison_n}")
ax.legend()
plt.show()

## Part 6: What Monte Carlo uncertainty means

Three distinctions will matter throughout Unit 4:

| Source of variation | Question answered | Reduced by more Monte Carlo draws? |
|---|---|---|
| Monte Carlo sampling error | How accurately did the algorithm estimate an expectation? | Yes |
| Experimental shot noise | How much would repeated physical data sets vary? | Only by collecting more physical shots |
| Parameter or calibration uncertainty | Which model inputs remain uncertain? | No; it must be measured or inferred |

A large number of simulated samples can make the first row tiny while leaving the other two unchanged. Reporting only the number of Monte Carlo draws is therefore not a complete uncertainty statement.

### Choosing a sampling strategy

| Method | Main idea | Strength | Limitation |
|---|---|---|---|
| Direct Monte Carlo | Draw from the density defining the expectation | Simple and dimension-independent $1/\sqrt{N}$ scaling | Can waste samples where the integrand contributes little |
| Stratified sampling | Force coverage of subregions | Often reduces variance for smooth functions | More bookkeeping in many dimensions |
| Importance sampling | Draw from a proposal concentrated in important regions and reweight | Powerful for localized integrands and rare events | Poor proposals produce extreme weights |
| Quasi-Monte Carlo | Use low-discrepancy deterministic points | Often converges faster for smooth moderate-dimensional problems | Error estimation and high-dimensional performance require care |

No method removes the need for a convergence study. Increase the sample count, repeat with independent randomizations, and verify that the observed spread is compatible with the reported Monte Carlo error.

## References and further reading

- [Unit 4 course resources](../resources/unit-4-monte-carlo-inference.md)
- [NumPy random `Generator` documentation](https://numpy.org/doc/stable/reference/random/generator.html)
- [SciPy quasi-Monte Carlo documentation](https://docs.scipy.org/doc/scipy/reference/stats.qmc.html)
- [Allen Downey, *Think Bayes 2*](https://allendowney.github.io/ThinkBayes2/)
- [Mark Newman, *Computational Physics* resources](https://websites.umich.edu/~mejn/cp2/)